In [ ]:
%pip install census us

In [ ]:
import requests

url = "https://legislature.mi.gov/Bills/Bill?ObjectName=2025-SB-0001"

# Disguise the request as a real browser to get past basic bot filtering
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

resp = requests.get(url, headers=headers, timeout=30)

# Access-check signals: status code, whether the page's unique markers are present
status = resp.status_code
has_heading = "BillHeading" in resp.text
has_content = "Jeremy Moss" in resp.text

status, len(resp.text), has_heading, has_content

In [ ]:
# Parse the fetched HTML and extract the three top-level fields: title, subject, primary sponsor
from bs4 import BeautifulSoup

soup = BeautifulSoup(resp.text, "html.parser")

# Title: <h1 id="BillHeading">
heading_el = soup.find(id="BillHeading")
title = heading_el.get_text(strip=True) if heading_el else None

# Subject (what the bill actually does): <div id="ObjectSubject">
subject_el = soup.find(id="ObjectSubject")
subject = subject_el.get_text(strip=True) if subject_el else None

# Primary sponsor: <a class="primarySponsor">
sponsor_el = soup.find("a", class_="primarySponsor")
primary_sponsor = sponsor_el.get_text(strip=True) if sponsor_el else None

title, primary_sponsor, subject


In [ ]:
# Extract multi-value fields: cosponsors (list) and action history (list of dicts)

# Cosponsors: all <a> inside #SponsorList except the primary sponsor
sponsor_links = soup.select("#SponsorList li a")
cosponsors = [
    a.get_text(strip=True)
    for a in sponsor_links
    if "primarySponsor" not in (a.get("class") or [])
]

# Actions: each <tr> in the History table has 3 <td>s (date, journal, description)
action_rows = soup.select("#History table tbody tr")
actions = []
for row in action_rows:
    cells = row.find_all("td")
    if len(cells) >= 3:
        actions.append({
            "date": cells[0].get_text(strip=True),
            "description": cells[2].get_text(strip=True),
        })

cosponsors, len(actions), actions[:3]

In [ ]:
# Find the URL of the bill's full text (the "Introduced Bill" HTML document)
from urllib.parse import urljoin

BASE = "https://legislature.mi.gov"

doc_rows = soup.select(".billDocuments .billDocRow")

text_url = None
for row in doc_rows:
    label_el = row.select_one(".text")
    label = label_el.get_text(strip=True) if label_el else ""
    html_link = row.select_one(".html a")
    if html_link and "Introduced Bill" in label:
        text_url = urljoin(BASE, html_link["href"])
        break

# Fallback: if no "Introduced Bill" row matched, take the first available html link
if text_url is None and doc_rows:
    first_link = doc_rows[0].select_one(".html a")
    if first_link:
        text_url = urljoin(BASE, first_link["href"])

text_url

In [ ]:
# Fetch the full bill text from text_url, then build content_hash from all actions
import hashlib

# Fetch and extract the full text of the bill document
text_resp = requests.get(text_url, headers=headers, timeout=30)
text_soup = BeautifulSoup(text_resp.text, "html.parser")
full_text = text_soup.get_text(separator="\n", strip=True)

# Build content_hash by joining ALL actions (order-independent change detection)
actions_joined = "\n".join(f"{a['date']} {a['description']}" for a in actions)
content_hash = hashlib.md5(actions_joined.encode()).hexdigest()

len(full_text), content_hash, full_text[:300]

In [ ]:
# Define scrape_bill(): fetch + parse one bill into a single dictionary
from datetime import datetime

def scrape_bill(bill_id):
    # Build the detail-page URL from the bill_id and fetch it
    detail_url = f"{BASE}/Bills/Bill?ObjectName={bill_id}"
    resp = requests.get(detail_url, headers=headers, timeout=30)
    soup = BeautifulSoup(resp.text, "html.parser")

    # Chamber is derivable from the bill_id (e.g. "2025-SB-0001" -> Senate)
    chamber = "Senate" if "-SB-" in bill_id else "House"

    # Top-level single-value fields
    heading_el = soup.find(id="BillHeading")
    title = heading_el.get_text(strip=True) if heading_el else None

    subject_el = soup.find(id="ObjectSubject")
    subject = subject_el.get_text(strip=True) if subject_el else None

    sponsor_el = soup.find("a", class_="primarySponsor")
    primary_sponsor = sponsor_el.get_text(strip=True) if sponsor_el else None

    # Cosponsors: all sponsor links except the primary one
    sponsor_links = soup.select("#SponsorList li a")
    cosponsors = [
        a.get_text(strip=True)
        for a in sponsor_links
        if "primarySponsor" not in (a.get("class") or [])
    ]

    # Action history: date + description from each table row
    actions = []
    for row in soup.select("#History table tbody tr"):
        cells = row.find_all("td")
        if len(cells) >= 3:
            actions.append({
                "date": cells[0].get_text(strip=True),
                "description": cells[2].get_text(strip=True),
            })

    # Locate the "Introduced Bill" HTML document link for the full text
    text_url = None
    doc_rows = soup.select(".billDocuments .billDocRow")
    for r in doc_rows:
        label_el = r.select_one(".text")
        label = label_el.get_text(strip=True) if label_el else ""
        link = r.select_one(".html a")
        if link and "Introduced Bill" in label:
            text_url = urljoin(BASE, link["href"])
            break
    if text_url is None and doc_rows:
        first_link = doc_rows[0].select_one(".html a")
        if first_link:
            text_url = urljoin(BASE, first_link["href"])

    # Fetch the full text if a document URL was found
    full_text = None
    if text_url:
        tr = requests.get(text_url, headers=headers, timeout=30)
        full_text = BeautifulSoup(tr.text, "html.parser").get_text(separator="\n", strip=True)

    # content_hash from all actions joined (order-independent change detection)
    actions_joined = "\n".join(f"{a['date']} {a['description']}" for a in actions)
    content_hash = hashlib.md5(actions_joined.encode()).hexdigest()

    # Assemble the single bill record
    return {
        "bill_id": bill_id,
        "chamber": chamber,
        "url": detail_url,
        "content_hash": content_hash,
        "last_scraped": datetime.now().isoformat(timespec="seconds"),
        "title": title,
        "subject": subject,
        "primary_sponsor": primary_sponsor,
        "cosponsors": cosponsors,
        "actions": actions,
        "text_url": text_url,
        "full_text": full_text,
    }

In [ ]:
# Test scrape_bill() on a single known bill
record = scrape_bill("2025-SB-0001")
{k: record[k] for k in ["bill_id", "chamber", "content_hash", "title", "primary_sponsor"]} , len(record["cosponsors"]), len(record["actions"]), len(record["full_text"])

In [ ]:
# Diagnose why full_text is None: check text_url and the raw text response separately
print("text_url:", record["text_url"])

if record["text_url"]:
    diag = requests.get(record["text_url"], headers=headers, timeout=30)
    print("status:", diag.status_code)
    print("length:", len(diag.text))
    print("preview:", diag.text[:200])

In [ ]:
# Diagnose: re-fetch the detail page fresh and check whether the Documents section exists in it
detail_url = f"{BASE}/Bills/Bill?ObjectName=2025-SB-0001"
d = requests.get(detail_url, headers=headers, timeout=30)

print("status:", d.status_code)
print("length:", len(d.text))
print("has billDocuments?:", "billDocuments" in d.text)
print("has BillHeading?:", "BillHeading" in d.text)

In [ ]:
# Define fetch(): a single request helper with polite delay + retry to survive bot blocking
import time

def fetch(url, delay=2.0, retries=3, backoff=5.0):
    # Polite pause before every request to reduce load and avoid triggering blocks
    time.sleep(delay)
    last_err = None
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=headers, timeout=60)
            resp.raise_for_status()  # treat 4xx/5xx as errors so they get retried
            return resp
        except Exception as e:
            last_err = e
            # Wait longer on each failed attempt before retrying
            time.sleep(backoff * (attempt + 1))
    # All retries exhausted: re-raise so the caller can log it
    raise last_err

In [ ]:
# Test fetch() on the detail page and confirm the polite delay is applied
start = time.time()
resp = fetch("https://legislature.mi.gov/Bills/Bill?ObjectName=2025-SB-0001")
elapsed = time.time() - start

resp.status_code, len(resp.text), round(elapsed, 1)

In [ ]:
# Redefine fetch() with a reused Session, longer randomized delay, and retries
import random

# One Session reused across all requests: keeps connection/cookies, looks less bot-like
session = requests.Session()
session.headers.update(headers)

def fetch(url, min_delay=5.0, max_delay=7.0, retries=3, backoff=10.0):
    # Randomized polite pause before each request to avoid mechanical request patterns
    time.sleep(random.uniform(min_delay, max_delay))
    last_err = None
    for attempt in range(retries):
        try:
            resp = session.get(url, timeout=60)
            resp.raise_for_status()
            return resp
        except Exception as e:
            last_err = e
            # Longer wait on each failed attempt before retrying
            time.sleep(backoff * (attempt + 1))
    raise last_err

In [ ]:
# Single cautious test of the hardened fetch()
resp = fetch("https://legislature.mi.gov/Bills/Bill?ObjectName=2025-SB-0001")
resp.status_code, len(resp.text), "BillHeading" in resp.text

In [ ]:
# Redefine scrape_bill() to route every request through the hardened fetch()
def scrape_bill(bill_id):
    # Build the detail-page URL and fetch it via the hardened fetch()
    detail_url = f"{BASE}/Bills/Bill?ObjectName={bill_id}"
    resp = fetch(detail_url)
    soup = BeautifulSoup(resp.text, "html.parser")

    # Chamber is derivable from the bill_id (e.g. "2025-SB-0001" -> Senate)
    chamber = "Senate" if "-SB-" in bill_id else "House"

    # Top-level single-value fields
    heading_el = soup.find(id="BillHeading")
    title = heading_el.get_text(strip=True) if heading_el else None

    subject_el = soup.find(id="ObjectSubject")
    subject = subject_el.get_text(strip=True) if subject_el else None

    sponsor_el = soup.find("a", class_="primarySponsor")
    primary_sponsor = sponsor_el.get_text(strip=True) if sponsor_el else None

    # Cosponsors: all sponsor links except the primary one
    sponsor_links = soup.select("#SponsorList li a")
    cosponsors = [
        a.get_text(strip=True)
        for a in sponsor_links
        if "primarySponsor" not in (a.get("class") or [])
    ]

    # Action history: date + description from each table row
    actions = []
    for row in soup.select("#History table tbody tr"):
        cells = row.find_all("td")
        if len(cells) >= 3:
            actions.append({
                "date": cells[0].get_text(strip=True),
                "description": cells[2].get_text(strip=True),
            })

    # Locate the "Introduced Bill" HTML document link for the full text
    text_url = None
    doc_rows = soup.select(".billDocuments .billDocRow")
    for r in doc_rows:
        label_el = r.select_one(".text")
        label = label_el.get_text(strip=True) if label_el else ""
        link = r.select_one(".html a")
        if link and "Introduced Bill" in label:
            text_url = urljoin(BASE, link["href"])
            break
    if text_url is None and doc_rows:
        first_link = doc_rows[0].select_one(".html a")
        if first_link:
            text_url = urljoin(BASE, first_link["href"])

    # Fetch the full text via fetch() if a document URL was found
    full_text = None
    if text_url:
        tr = fetch(text_url)
        full_text = BeautifulSoup(tr.text, "html.parser").get_text(separator="\n", strip=True)

    # content_hash from all actions joined (order-independent change detection)
    actions_joined = "\n".join(f"{a['date']} {a['description']}" for a in actions)
    content_hash = hashlib.md5(actions_joined.encode()).hexdigest()

    # Assemble the single bill record
    return {
        "bill_id": bill_id,
        "chamber": chamber,
        "url": detail_url,
        "content_hash": content_hash,
        "last_scraped": datetime.now().isoformat(timespec="seconds"),
        "title": title,
        "subject": subject,
        "primary_sponsor": primary_sponsor,
        "cosponsors": cosponsors,
        "actions": actions,
        "text_url": text_url,
        "full_text": full_text,
    }

In [ ]:
# Test the fetch-powered scrape_bill() on a single bill
record = scrape_bill("2025-SB-0001")
(
    record["bill_id"],
    record["content_hash"],
    record["primary_sponsor"],
    len(record["cosponsors"]),
    len(record["actions"]),
    len(record["full_text"]) if record["full_text"] else None,
)

In [ ]:
# Fetch the Senate bill list page and check whether all bill_ids are present statically
import re

list_url = "https://legislature.mi.gov/Search/ExecuteSearch?sessions=2025-2026&docTypes=Senate%20Bill"
resp = fetch(list_url)

# Find every detail-page reference like ObjectName=2025-SB-0001 in the raw HTML
found = re.findall(r"ObjectName=(20\d\d-SB-\d+)", resp.text)
unique_ids = sorted(set(found))

len(resp.text), len(found), len(unique_ids), unique_ids[:5], unique_ids[-5:]

In [ ]:
# Diagnose how bill links/ids are actually represented in the 762KB search response
text = resp.text

# Count several candidate patterns to see which representation the page uses
candidates = {
    "ObjectName= (any)": len(re.findall(r"ObjectName=", text)),
    "objectName= (lower o)": len(re.findall(r"objectName=", text)),
    "/Bills/Bill": len(re.findall(r"/Bills/Bill", text)),
    "SB followed by digits": len(re.findall(r"SB[\s-]?\d+", text)),
    "GetObject": len(re.findall(r"GetObject", text)),
}

# Grab the HTML around the first occurrence of "SB" to see the actual markup
idx = text.find("SB")
snippet = text[idx-200:idx+200] if idx != -1 else "no 'SB' found"

candidates, snippet

In [ ]:
# Extract all Senate bill_ids using the correct lowercase objectName= pattern
found = re.findall(r"objectName=(20\d\d-SB-\d+)", resp.text)
senate_ids = sorted(set(found))

len(found), len(senate_ids), senate_ids[:5], senate_ids[-5:]

In [ ]:
# Define collect_bill_ids(): gather all Senate + House bill_ids from the search list pages
def collect_bill_ids():
    # Each chamber differs only by the docTypes query value and the id code (SB/HB)
    chambers = [
        {"doc_type": "Senate%20Bill", "code": "SB"},
        {"doc_type": "House%20Bill", "code": "HB"},
    ]

    all_ids = []
    for ch in chambers:
        url = f"{BASE}/Search/ExecuteSearch?sessions=2025-2026&docTypes={ch['doc_type']}"
        r = fetch(url)
        # Extract ids for this chamber's code only (e.g. 2025-SB-0001)
        ids = re.findall(rf"objectName=(20\d\d-{ch['code']}-\d+)", r.text)
        all_ids.extend(sorted(set(ids)))

    return all_ids

In [ ]:
# Run the collector and check totals for both chambers
bill_ids = collect_bill_ids()
senate = [b for b in bill_ids if "-SB-" in b]
house = [b for b in bill_ids if "-HB-" in b]

len(bill_ids), len(senate), len(house), house[:3], house[-3:]

### Narrowing down to certain subjects
using time to avoid ban, it is estimated to take more than 11 hours to scrap all the bills in both the house and the senate.

In [ ]:
# Count bills per category (2025-2026) to pick a focused, high-activity topic by data
from urllib.parse import quote

# Interest-aligned categories + a few high-activity comparators
categories = [
    "Environmental protection", "Natural resources", "Energy", "Water supply",
    "Women", "Family law",
    "Children", "Juveniles",
    "Labor", "Employment security", "Worker's compensation",
    "Health", "Education", "Crimes", "Elections",
]

category_counts = {}
for cat in categories:
    url = f"{BASE}/Search/ExecuteSearch?sessions=2025-2026&docTypes=Bills&category={quote(cat)}"
    r = fetch(url)
    ids = re.findall(r"objectName=(20\d\d-(?:SB|HB)-\d+)", r.text)
    category_counts[cat] = len(set(ids))

# Sort by count so the most active topics surface first
dict(sorted(category_counts.items(), key=lambda x: x[1], reverse=True))

### reduced to labor-related categories

In [ ]:
# Redefine collect_bill_ids(): union of bill_ids across the three labor-related categories
def collect_bill_ids():
    # Labor topic defined as the union of these three categories
    categories = ["Labor", "Employment security", "Worker's compensation"]

    all_ids = set()
    for cat in categories:
        url = f"{BASE}/Search/ExecuteSearch?sessions=2025-2026&docTypes=Bills&category={quote(cat)}"
        r = fetch(url)
        ids = set(re.findall(r"objectName=(20\d\d-(?:SB|HB)-\d+)", r.text))
        all_ids |= ids  # union across categories, duplicates removed automatically

    return sorted(all_ids)

In [ ]:
# Run the labor-topic collector and check the total and chamber split
bill_ids = collect_bill_ids()
senate = [b for b in bill_ids if "-SB-" in b]
house = [b for b in bill_ids if "-HB-" in b]

len(bill_ids), len(senate), len(house), bill_ids[:5]

In [ ]:
# Define initial_scrape(): loop over all bill_ids, scrape each, log successes and failures
def initial_scrape(bill_ids):
    data = {}          # bill_id -> scraped record
    change_log = []    # what changed this run (all "added" on the initial run)
    error_log = []     # which bills failed and why

    for i, bill_id in enumerate(bill_ids, start=1):
        try:
            record = scrape_bill(bill_id)
            data[bill_id] = record
            change_log.append({"bill_id": bill_id, "change": "added"})
        except Exception as e:
            # Log the failure and keep going; one bad page must not stop the whole run
            error_log.append({"bill_id": bill_id, "error": str(e)})

        # Lightweight progress marker for this long-running loop
        print(f"[{i}/{len(bill_ids)}] {bill_id}")

    return data, change_log, error_log

In [ ]:
# Smoke-test the loop on just the first 3 bills before the full 20-minute run
test_data, test_changes, test_errors = initial_scrape(bill_ids[:3])
len(test_data), len(test_changes), len(test_errors), list(test_data.keys())

In [ ]:
# Run the full initial scrape over all 160 bills, then save data + logs to JSON files
import json

# This runs the full ~20-minute scrape (160 bills x 2 requests each, with delays)
data, change_log, error_log = initial_scrape(bill_ids)

# Persist the core database and this run's logs
with open("data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

with open("change_log.json", "w", encoding="utf-8") as f:
    json.dump(change_log, f, ensure_ascii=False, indent=2)

with open("error_log.json", "w", encoding="utf-8") as f:
    json.dump(error_log, f, ensure_ascii=False, indent=2)

len(data), len(error_log)

In [ ]:
# Hardened fetch (8-12s delay) + resumable/checkpointed scrape_all + first 30-bill batch

def fetch(url, min_delay=8.0, max_delay=12.0, retries=3, backoff=10.0):
    # Longer randomized polite pause before each request to reduce bot-blocking
    time.sleep(random.uniform(min_delay, max_delay))
    last_err = None
    for attempt in range(retries):
        try:
            resp = session.get(url, timeout=60)
            resp.raise_for_status()
            return resp
        except Exception as e:
            last_err = e
            time.sleep(backoff * (attempt + 1))
    raise last_err

def load_json(path, default):
    # Read an existing JSON file, or return a default if it doesn't exist yet
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return default

def save_json(path, obj):
    # Write an object to a JSON file (used as a checkpoint after each bill)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def scrape_all(bill_ids):
    # Resume from whatever was already saved to disk
    data = load_json("data.json", {})
    error_log = load_json("error_log.json", [])

    for i, bill_id in enumerate(bill_ids, start=1):
        # Resume: skip bills already scraped
        if bill_id in data:
            print(f"[{i}/{len(bill_ids)}] {bill_id} (skip)")
            continue
        try:
            data[bill_id] = scrape_bill(bill_id)
            save_json("data.json", data)        # checkpoint after each bill
            print(f"[{i}/{len(bill_ids)}] {bill_id} (saved)")
        except Exception as e:
            error_log.append({"bill_id": bill_id, "error": str(e)})
            save_json("error_log.json", error_log)
            print(f"[{i}/{len(bill_ids)}] {bill_id} (ERROR)")

    return data, error_log

# Run the first batch of 30 bills (safe to re-run: resume + checkpoint)
data, error_log = scrape_all(bill_ids[:30])
len(data), len(error_log)

### Using Proxy from here (ScrapeOps)

In [ ]:
# Load the ScrapeOps API key from a .env file (kept out of git via .gitignore)
import os
from dotenv import load_dotenv

load_dotenv()
SCRAPEOPS_API_KEY = os.getenv("SCRAPEOPS_API_KEY")

# Sanity check: confirm the key loaded without printing it
SCRAPEOPS_API_KEY is not None

In [ ]:
# Redefine fetch() to route all requests through the ScrapeOps proxy
from urllib.parse import urlencode

def fetch(url, min_delay=1.0, max_delay=2.0, retries=3, backoff=10.0):
    # Short pause now that the proxy rotates IPs for us
    time.sleep(random.uniform(min_delay, max_delay))

    # Wrap the target URL inside the ScrapeOps proxy request
    proxy_endpoint = "https://proxy.scrapeops.io/v1/"
    params = {"api_key": SCRAPEOPS_API_KEY, "url": url}

    last_err = None
    for attempt in range(retries):
        try:
            # Proxy can be slow (15-20s per request), so allow a long timeout
            resp = session.get(proxy_endpoint, params=params, timeout=120)
            resp.raise_for_status()
            return resp
        except Exception as e:
            last_err = e
            time.sleep(backoff * (attempt + 1))
    raise last_err

In [ ]:
# Single test: confirm the proxy successfully fetches one bill detail page
resp = fetch(f"{BASE}/Bills/Bill?ObjectName=2025-SB-0001")
resp.status_code, len(resp.text), "BillHeading" in resp.text

In [ ]:
# Scrape the first 40 bills via proxy (resume skips already-saved ones; checkpoint saves each)
data, error_log = scrape_all(bill_ids[:40])
len(data), len(error_log)

In [ ]:
# Continue: scrape up to bill 90 (already-saved ones are skipped)
data, error_log = scrape_all(bill_ids[:90])
len(data), len(error_log)

In [ ]:
# Continue: scrape up to bill 130 (already-saved ones are skipped)
data, error_log = scrape_all(bill_ids[:130])
len(data), len(error_log)

In [ ]:
# Final batch: scrape all remaining bills (already-saved ones are skipped)
data, error_log = scrape_all(bill_ids)
len(data), len(error_log)

### check the three errors

In [ ]:
# Inspect the 3 failed bills to see whether they're retryable
error_log

In [ ]:
# Quality check: how many records have full_text, sponsor, actions populated
saved = load_json("data.json", {})
has_text = sum(1 for b in saved.values() if b.get("full_text"))
has_sponsor = sum(1 for b in saved.values() if b.get("primary_sponsor"))
has_actions = sum(1 for b in saved.values() if b.get("actions"))

len(saved), has_text, has_sponsor, has_actions

In [ ]:
# Clean error_log: drop entries whose bill_id was actually scraped successfully into data
saved = load_json("data.json", {})
error_log = load_json("error_log.json", [])

# Keep only errors for bills that are genuinely missing from data
real_errors = [e for e in error_log if e["bill_id"] not in saved]
save_json("error_log.json", real_errors)

len(real_errors)

In [ ]:
# Define get_content_hash(): fetch only the detail page and compute the actions-based hash
def get_content_hash(bill_id):
    detail_url = f"{BASE}/Bills/Bill?ObjectName={bill_id}"
    resp = fetch(detail_url)
    soup = BeautifulSoup(resp.text, "html.parser")

    # Extract actions the same way scrape_bill does
    actions = []
    for row in soup.select("#History table tbody tr"):
        cells = row.find_all("td")
        if len(cells) >= 3:
            actions.append({
                "date": cells[0].get_text(strip=True),
                "description": cells[2].get_text(strip=True),
            })

    # Same hashing logic as scrape_bill (join all actions, order-independent)
    actions_joined = "\n".join(f"{a['date']} {a['description']}" for a in actions)
    return hashlib.md5(actions_joined.encode()).hexdigest()

In [ ]:
# Test: the freshly computed hash should match the stored hash for an unchanged bill
saved = load_json("data.json", {})
test_id = "2025-SB-0001"
saved.get(test_id, {}).get("content_hash"), get_content_hash(test_id)

In [ ]:
# Diagnose the 401: read ScrapeOps' actual error message
r = session.get(
    "https://proxy.scrapeops.io/v1/",
    params={"api_key": SCRAPEOPS_API_KEY, "url": f"{BASE}/Bills/Bill?ObjectName=2025-SB-0001"},
    timeout=120,
)
print("status:", r.status_code)
print("body:", r.text[:500])

the initially issued API is still overriding; the code below make the new one overrule

In [ ]:
# Reload .env and check the key prefix only (to verify the update without exposing the full key)
load_dotenv(override=True)  # override=True forces reload of changed .env values
SCRAPEOPS_API_KEY = os.getenv("SCRAPEOPS_API_KEY")
SCRAPEOPS_API_KEY[:8] if SCRAPEOPS_API_KEY else None

In [ ]:
# Test: the freshly computed hash should match the stored hash for an unchanged bill
saved = load_json("data.json", {})
test_id = "2025-SB-0001"
saved.get(test_id, {}).get("content_hash"), get_content_hash(test_id)

In [ ]:
# Test hash consistency on a bill that actually exists in our labor dataset
saved = load_json("data.json", {})
test_id = bill_ids[0]  # a real labor bill we already scraped
saved[test_id]["content_hash"], get_content_hash(test_id)

In [ ]:
# Define update_scrape(): detect new/modified labor bills via hash and refresh data + logs
def update_scrape():
    data = load_json("data.json", {})
    error_log = []                     # errors for this update run
    change_log = []                    # changes detected this update run
    run_ts = datetime.now().isoformat(timespec="seconds")

    # Re-fetch the current labor bill list so newly introduced bills are caught
    current_ids = collect_bill_ids()

    for i, bill_id in enumerate(current_ids, start=1):
        try:
            if bill_id not in data:
                # New bill introduced since the last run
                data[bill_id] = scrape_bill(bill_id)
                change_log.append({"bill_id": bill_id, "change": "added", "run": run_ts})
                save_json("data.json", data)
                print(f"[{i}/{len(current_ids)}] {bill_id} (added)")
            else:
                # Existing bill: compare hashes to see if anything changed
                new_hash = get_content_hash(bill_id)
                if new_hash != data[bill_id]["content_hash"]:
                    data[bill_id] = scrape_bill(bill_id)
                    change_log.append({"bill_id": bill_id, "change": "modified", "run": run_ts})
                    save_json("data.json", data)
                    print(f"[{i}/{len(current_ids)}] {bill_id} (modified)")
                else:
                    print(f"[{i}/{len(current_ids)}] {bill_id} (skip)")
        except Exception as e:
            error_log.append({"bill_id": bill_id, "error": str(e), "run": run_ts})
            save_json("error_log.json", error_log)
            print(f"[{i}/{len(current_ids)}] {bill_id} (ERROR)")

    # Save this run's change log
    save_json("change_log.json", change_log)
    return data, change_log, error_log

In [ ]:
# Test version of update_scrape that takes an explicit target list (to test on a few bills)
def update_scrape_targets(target_ids):
    data = load_json("data.json", {})
    error_log = []
    change_log = []
    run_ts = datetime.now().isoformat(timespec="seconds")

    for i, bill_id in enumerate(target_ids, start=1):
        try:
            if bill_id not in data:
                data[bill_id] = scrape_bill(bill_id)
                change_log.append({"bill_id": bill_id, "change": "added", "run": run_ts})
                save_json("data.json", data)
                print(f"[{i}/{len(target_ids)}] {bill_id} (added)")
            else:
                new_hash = get_content_hash(bill_id)
                if new_hash != data[bill_id]["content_hash"]:
                    data[bill_id] = scrape_bill(bill_id)
                    change_log.append({"bill_id": bill_id, "change": "modified", "run": run_ts})
                    save_json("data.json", data)
                    print(f"[{i}/{len(target_ids)}] {bill_id} (modified)")
                else:
                    print(f"[{i}/{len(target_ids)}] {bill_id} (skip)")
        except Exception as e:
            error_log.append({"bill_id": bill_id, "error": str(e), "run": run_ts})
            print(f"[{i}/{len(target_ids)}] {bill_id} (ERROR)")

    return change_log, error_log

In [ ]:
# Test the update logic on 3 already-scraped bills (all should report "skip")
changes, errors = update_scrape_targets(bill_ids[:3])
changes, errors